# Category example structures

Visualize representative generated, matched training, and relaxed substituted training structures for the novelty categories used in `classification.ipynb`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatviz import structure_2d

import __main__

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    notebooks_dir = path / "notebooks"
    if (notebooks_dir / "notebook_utils.py").exists():
        if str(notebooks_dir) not in sys.path:
            sys.path.insert(0, str(notebooks_dir))
        break
else:
    raise RuntimeError("Could not find notebooks directory")

from notebook_utils import find_repo_root  # noqa: E402

ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
SCRIPTS_DIR = ROOT / "scripts"
for import_path in (ROOT, NOTEBOOKS_DIR, SCRIPTS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from notebook_constants import (  # noqa: E402
    CATEGORY_LABELS,
    CATEGORY_ORDER,
    SUBSET_OPTIONS,
)
from notebook_utils import (  # noqa: E402
    classify_model,
    entries_to_frame,
    load_direct_matches,
    load_pickle_gz,
    missing_required_paths,
    required_paths,
)
from substitute_structures import SubstitutedEntry  # noqa: E402

from src.config import INPUT_DIR, RAW_RESULTS_DIR  # noqa: E402

# Some substituted-entry pickles were written with SubstitutedEntry resolved
# from __main__. Provide that symbol before unpickling.
__main__.SubstitutedEntry = SubstitutedEntry

ROOT, INPUT_DIR, RAW_RESULTS_DIR

In [ ]:
MODEL = "mattergen"
SUBSET = "metastable_smact_valid"
N_EXAMPLES_PER_CATEGORY = 5

if SUBSET not in SUBSET_OPTIONS:
    raise ValueError(f"SUBSET must be one of {sorted(SUBSET_OPTIONS)}, got {SUBSET!r}")
if N_EXAMPLES_PER_CATEGORY < 1:
    raise ValueError("N_EXAMPLES_PER_CATEGORY must be at least 1")

In [ ]:
CATEGORY_EXAMPLE_PATH_KEYS = (
    "generated_structures",
    "training_structures",
    "relaxed_ehull",
    "smact_validity",
    "direct_sm",
    "relaxed_sm_anon_entries",
    "relaxed_wyckoff_entries",
    "relaxed_sm_anon_matches",
    "relaxed_wyckoff_matches",
)


def model_required_paths(model: str) -> dict[str, Path]:
    return required_paths(model, INPUT_DIR, RAW_RESULTS_DIR, CATEGORY_EXAMPLE_PATH_KEYS)


paths = model_required_paths(MODEL)
missing_paths = pd.DataFrame(missing_required_paths(paths))
if not missing_paths.empty:
    display(missing_paths)
    raise FileNotFoundError(
        f"Missing required files for MODEL={MODEL!r}. "
        "Check INPUT_DIR and RAW_RESULTS_DIR."
    )

In [ ]:
classifications = classify_model(
    MODEL,
    paths,
    include_category_label=True,
)


def subset_classifications(classifications: pd.DataFrame, subset: str) -> pd.DataFrame:
    if subset == "all":
        return classifications.copy()
    if subset == "metastable":
        return classifications[classifications["is_metastable"]].copy()
    if subset == "metastable_smact_valid":
        return classifications[classifications["is_metastable_smact_valid"]].copy()
    raise ValueError(f"Unknown subset: {subset!r}")


selected_classifications = subset_classifications(classifications, SUBSET)
category_counts = (
    selected_classifications["category"]
    .value_counts()
    .reindex(CATEGORY_ORDER, fill_value=0)
    .rename_axis("category")
    .reset_index(name="count")
)
category_counts["category_label"] = category_counts["category"].map(CATEGORY_LABELS)

display(
    Markdown(
        f"**Model:** `{MODEL}`  \n"
        f"**Subset:** `{SUBSET}`  \n"
        f"**Selected samples:** {len(selected_classifications):,}"
    )
)
category_counts

In [ ]:
def load_relaxed_candidates(model_paths: dict[str, Path]) -> pd.DataFrame:
    tables = [
        entries_to_frame(
            load_pickle_gz(model_paths["relaxed_sm_anon_entries"]),
            load_pickle_gz(model_paths["relaxed_sm_anon_matches"]),
            "anon",
            include_structure=True,
        ),
        entries_to_frame(
            load_pickle_gz(model_paths["relaxed_wyckoff_entries"]),
            load_pickle_gz(model_paths["relaxed_wyckoff_matches"]),
            "wyckoff",
            include_structure=True,
        ),
    ]
    return pd.concat(tables, ignore_index=True)


def best_relaxed_candidates(candidates: pd.DataFrame) -> pd.DataFrame:
    matched = candidates[candidates["match"]].copy()
    if matched.empty:
        return matched
    matched = matched.sort_values(
        [
            "gen_idx",
            "cost_mod_petti",
            "cost_uniform",
            "source_order",
            "rank",
            "train_idx",
            "entry_idx",
        ],
        kind="mergesort",
    )
    return matched.drop_duplicates("gen_idx", keep="first").set_index("gen_idx")


relaxed_candidates = load_relaxed_candidates(paths)
best_candidates = best_relaxed_candidates(relaxed_candidates)
direct_train = (
    load_direct_matches(paths["direct_sm"])
    .sort_values(["gen_idx", "train_idx"], kind="mergesort")
    .drop_duplicates("gen_idx", keep="first")
    .set_index("gen_idx")["train_idx"]
)

pd.DataFrame(
    {
        "direct_match_gen_count": [len(direct_train)],
        "relaxed_match_gen_count": [len(best_candidates)],
    }
)

In [ ]:
EXAMPLE_COLUMNS = [
    "model",
    "gen_idx",
    "category",
    "category_label",
    "is_direct_sm_match",
    "has_relaxed_sm_anon_match",
    "has_relaxed_wyckoff_match",
    "ehull_relaxed",
    "is_metastable",
    "is_smact_valid",
    "is_metastable_smact_valid",
    "train_idx",
    "source",
    "rank",
    "cost_uniform",
    "cost_mod_petti",
    "relaxed_substituted_structure",
]


def select_examples(classifications: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for category in CATEGORY_ORDER:
        pool = classifications[classifications["category"] == category].sort_values(
            "gen_idx", kind="mergesort"
        )
        if category == "1":
            pool = pool[pool["gen_idx"].isin(direct_train.index)]
        elif category.startswith("2"):
            pool = pool[pool["gen_idx"].isin(best_candidates.index)]

        for _, row in pool.head(N_EXAMPLES_PER_CATEGORY).iterrows():
            gen_idx = int(row["gen_idx"])
            example = row.to_dict()
            example["train_idx"] = pd.NA
            example["source"] = pd.NA
            example["rank"] = pd.NA
            example["cost_uniform"] = pd.NA
            example["cost_mod_petti"] = pd.NA
            example["relaxed_substituted_structure"] = None

            if category == "1":
                example["train_idx"] = int(direct_train.loc[gen_idx])
            elif category.startswith("2"):
                candidate = best_candidates.loc[gen_idx]
                example["train_idx"] = int(candidate["train_idx"])
                example["source"] = str(candidate["source"])
                example["rank"] = int(candidate["rank"])
                example["cost_uniform"] = float(candidate["cost_uniform"])
                example["cost_mod_petti"] = float(candidate["cost_mod_petti"])
                example["relaxed_substituted_structure"] = candidate[
                    "relaxed_substituted_structure"
                ]

            rows.append(example)

    return pd.DataFrame(rows, columns=EXAMPLE_COLUMNS)


examples = select_examples(selected_classifications)
summary = (
    examples.groupby("category", observed=False)
    .size()
    .reindex(CATEGORY_ORDER, fill_value=0)
    .rename("selected_examples")
    .reset_index()
)
summary["category_label"] = summary["category"].map(CATEGORY_LABELS)
display(summary)
display(
    examples[
        [
            "category",
            "category_label",
            "gen_idx",
            "train_idx",
            "source",
            "rank",
            "cost_mod_petti",
            "cost_uniform",
            "ehull_relaxed",
            "is_smact_valid",
        ]
    ]
)

In [ ]:
generated_structures: list[Structure] = load_pickle_gz(paths["generated_structures"])
training_structures: list[Structure] = load_pickle_gz(paths["training_structures"])


def conventional_structure(structure: Structure) -> Structure:
    return SpacegroupAnalyzer(structure).get_conventional_standard_structure()


def structure_title(role: str, index_label: str, structure: Structure) -> str:
    analyzer = SpacegroupAnalyzer(structure)
    spg = f"{analyzer.get_space_group_symbol()} ({analyzer.get_space_group_number()})"
    formula = structure.composition.reduced_formula
    return f"{role}<br>{index_label}<br>{formula}<br>{spg}"


def format_cost(value: Any) -> str:
    if pd.isna(value):
        return "N/A"
    return f"{float(value):.6g}"


def plot_example(row: pd.Series) -> None:
    category = str(row["category"])
    gen_idx = int(row["gen_idx"])
    train_idx = None if pd.isna(row["train_idx"]) else int(row["train_idx"])

    structures: dict[str, Structure] = {
        "Generated": conventional_structure(generated_structures[gen_idx])
    }
    titles = {
        "Generated": structure_title(
            "Generated sample", f"gen_idx={gen_idx}", structures["Generated"]
        )
    }

    missing_roles = []
    if train_idx is None:
        missing_roles.append("matched train sample")
    else:
        structures["Train"] = conventional_structure(training_structures[train_idx])
        titles["Train"] = structure_title(
            "Matched train sample", f"train_idx={train_idx}", structures["Train"]
        )

    relaxed_substituted = row["relaxed_substituted_structure"]
    if relaxed_substituted is None:
        missing_roles.append("relaxed substituted train sample")
    else:
        structures["Relaxed substituted train"] = conventional_structure(
            relaxed_substituted
        )
        titles["Relaxed substituted train"] = structure_title(
            "Relaxed substituted train",
            f"{row['source']}, rank={int(row['rank'])}",
            structures["Relaxed substituted train"],
        )

    train_label = train_idx if train_idx is not None else "N/A"
    source_label = row["source"] if not pd.isna(row["source"]) else "N/A"
    missing_text = ""
    if missing_roles:
        missing_text = "  \nN/A: " + ", ".join(missing_roles)
    display(
        Markdown(
            f"### Category {category}: {CATEGORY_LABELS[category]}  \n"
            f"gen_idx=`{gen_idx}`; train_idx=`{train_label}`; "
            f"source=`{source_label}`; "
            f"mod-Pettifor cost=`{format_cost(row['cost_mod_petti'])}`; "
            f"uniform cost=`{format_cost(row['cost_uniform'])}`"
            f"{missing_text}"
        )
    )

    fig = structure_2d(
        structures,
        n_cols=3,
        show_cell=True,
        site_labels="legend",
        standardize_struct=False,
        subplot_title=lambda _struct, key: titles[key],
    )
    fig.update_layout(height=360, margin={"l": 10, "r": 10, "t": 80, "b": 10})
    display(fig)


if examples.empty:
    display(Markdown("No examples available for the selected model and subset."))
else:
    for _, example in examples.iterrows():
        plot_example(example)